In [32]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from scipy.stats import uniform, loguniform

from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV, train_test_split
from sklearn.metrics import make_scorer, brier_score_loss
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss
from scipy.stats import uniform

import xgboost as xgb

In [38]:
import pandas as pd
import os

# Assume root directory is './TrainingData'
root_dir = './TrainingData'

TRAIN_list = []
LABELS_list = []
fold_indices = []

SALE_COLUMNS = ["Actual Loss Calculation", "Zero Balance Removal UPB", "Net Sales Proceeds", 
                "Delinquent Accrued Interest", "Expenses", "MI Recoveries", "Non MI Recoveries"]

MISC_COLUMNS_TO_DROP = ["MSA", 'Postal Code', 'Major Stress', "Monthly Reporting Period",
                        'Distress Date', 'Zero Balance Code', 'Unresolved', 'Last Time Current',
                         'Property State', 'Current Loan Delinquency Status', 'Loan Sequence Number']
CAT_COLUMNS = []

fold = 0 

# Load in all files from TrainingData Folder
for year_folder in sorted(os.listdir(root_dir)):
    year_path = os.path.join(root_dir, year_folder)
    if os.path.isdir(year_path):
        for file in os.listdir(year_path):
            if file.endswith('.parquet'):
                file_path = os.path.join(year_path, file)
                df = pd.read_parquet(file_path)
                #df = df[(df["Major Stress"] == 1) & ~(df["Unresolved"] == 1)]
                TRAIN_list.append(df.drop(columns=['Default Flag'])) 
                LABELS_list.append(df['Default Flag'])               
                fold_indices.extend([fold] * len(df))
        fold += 1

TRAIN = pd.concat(TRAIN_list).reset_index(drop=True)

TRAIN[TRAIN["HPI Change %"].isna()]


#TRAIN = TRAIN.drop(columns=SALE_COLUMNS)


,Credit Score,First Time Homebuyer Flag,MSA,Mortgage Insurance Percentage (MI %),Number of Units,Occupancy Status,Original Combined Loan-to-Value (CLTV),Original Debt-to-Income (DTI) Ratio,Original UPB,Original Loan-to-Value (LTV),...,Zero Balance Code,Unresolved,Last Time Current,Actual Loss Calculation,Zero Balance Removal UPB,Net Sales Proceeds,Delinquent Accrued Interest,Expenses,MI Recoveries,Non MI Recoveries
870,629,N,NaN,0,1,P,74,31,140000,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3593,728,N,NaN,0,4,I,70,40,125000,70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7985,703,N,NaN,0,1,P,18,50,21000,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7988,678,N,NaN,0,1,P,34,46,46000,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12202,663,N,NaN,0,1,P,74,37,114000,74,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1408822,809,Y,NaN,0,1,P,46,14,300000,46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1408832,650,N,NaN,0,1,P,70,38,350000,70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1409289,694,N,41980.0,0,1,S,70,36,151000,70,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1409473,690,N,NaN,0,1,P,80,42,165000,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
TRAIN = TRAIN.drop(columns=MISC_COLUMNS_TO_DROP)
categorical_cols = TRAIN.select_dtypes(include=['object', 'category']).columns.tolist()

TRAIN = pd.get_dummies(TRAIN, columns=categorical_cols, drop_first=True)
LABELS = pd.concat(LABELS_list).reset_index(drop=True)
LABELS.fillna(0, inplace=True)

In [34]:
TRAIN[TRAIN["HPI Change %"].isna()]

,Credit Score,First Time Homebuyer Flag,MSA,Mortgage Insurance Percentage (MI %),Number of Units,Occupancy Status,Original Combined Loan-to-Value (CLTV),Original Debt-to-Income (DTI) Ratio,Original UPB,Original Loan-to-Value (LTV),...,MaxPriorDelinquency,Current Unemployment Rate,Unemployment Rate at Origination,HPI Change %,Unemployment Rate Change 1Y,Distress Date,Major Stress,Zero Balance Code,Unresolved,Last Time Current
1156,676,N,NaN,0,1,P,84,999,104000,84,...,0.0,NaN,NaN,NaN,NaN,201601.0,1.0,NaN,0.0,201507.0
1292,691,N,NaN,35,1,P,110,999,241000,110,...,3.0,NaN,NaN,NaN,NaN,201610.0,1.0,NaN,0.0,201604.0
1332,710,N,NaN,35,1,I,96,999,130000,96,...,4.0,NaN,NaN,NaN,NaN,201612.0,1.0,NaN,0.0,201606.0
1480,587,N,NaN,0,1,P,208,999,222000,105,...,1.0,NaN,NaN,NaN,NaN,201709.0,1.0,NaN,0.0,202304.0
1728,755,N,NaN,0,1,P,80,29,269000,80,...,2.0,NaN,NaN,NaN,NaN,201810.0,1.0,NaN,0.0,202205.0
1982,691,N,NaN,0,1,P,74,36,93000,74,...,3.0,NaN,NaN,NaN,NaN,201710.0,1.0,NaN,0.0,201704.0


In [29]:
TRAIN["Estimated Value at Origination"] = TRAIN["Original UPB"] / TRAIN["Original Combined Loan-to-Value (CLTV)"]
TRAIN["Estimated Current Value"] = TRAIN["Estimated Value at Origination"] * (TRAIN["HPI Change %"] + 1)

TRAIN["Imputed Estimated LTV"] = TRAIN["Current Actual UPB"] / TRAIN["Estimated Current Value"]

In [30]:
from sklearn.feature_selection import mutual_info_classif

# X: your feature matrix
# y: your binary target

mi_scores = []
for i in range(TRAIN.shape[1]):  # loop over features
    Xi = TRAIN.iloc[:, i]        # <-- FIX here
    mask = ~Xi.isna()            # better Pandas NaN mask
    mi = mutual_info_classif(Xi[mask].values.reshape(-1, 1), LABELS[mask])
    mi_scores.append(mi[0])

In [31]:
mi_series = pd.Series(mi_scores, index=TRAIN.columns)

# Sort from most informative to least
mi_series = mi_series.sort_values(ascending=False)

print(mi_series)


Imputed Estimated LTV                          0.091520
Estimated Current Value                        0.068804
Original Combined Loan-to-Value (CLTV)         0.062944
Original Loan-to-Value (LTV)                   0.061606
Estimated Loan-to-Value (ELTV)                 0.060869
Estimated Value at Origination                 0.051512
Original Debt-to-Income (DTI) Ratio            0.049894
Numeric Delinquency                            0.040223
Loan Purpose_N                                 0.037807
Remaining Months to Legal Maturity             0.034934
Current Unemployment Rate                      0.027352
MaxPriorDelinquency                            0.026850
Program Indicator_F                            0.026131
Original Loan Term                             0.025384
Loan Purpose_P                                 0.025269
Current Actual UPB                             0.025144
HPI Change %                                   0.024306
Mortgage Insurance Cancellation Indicator_7    0